# Kiểm chứng numeric cho AGOPNs

Cell dưới đây gộp toàn bộ ba script kiểm chứng thành **một cell tự chứa**:

| Phần | Nội dung | Kết luận |
|------|----------|----------|
| **[A]** `verify_math` | `P` là **left factor** của `Δ̃`; `Mᵀh_b ≈ 0` (code) vs `Mh_b ≉ 0` (Eq.1 manuscript); `M` rank-1; bfloat16 phá triệt tiêu benign | code inference **đúng**, Eq.(1)/§Step4 **sai chiều** |
| **[B]** `verify_qspace` | đường Q-space ≡ đường ma trận cũ; `u ∈ range(Q)` theo cấu trúc; TF32 làm hỏng assert cũ; leakage + selectivity ở quy mô thật (d=4096) | lỗi *"P không idempotent"* = **TF32**; v4 khắc phục tận gốc |
| **[C]** `verify_ablation` | selectivity **FULL** vs **IDENTITY (P=I)** vs **VECTOR (h+αr)**; `λ` không dùng chung được giữa full & ablation | ablation `identity` là phép cô lập đúng null-space |

Cell tự import `src/utils/steering_utils.py` của repo (có fallback nếu chạy ngoài repo).
Đặt `DEV = "cuda"` trong cell nếu muốn kiểm chứng luôn trên GPU (TF32 đã được tắt tự động).


---
### [A] Left vs Right factor — vì sao code đúng còn manuscript sai

Với `A = P Hₘᵀ Hₘ P + λP` và `P² = P, Pᵀ = P` ta có `A = PAP ⇒ A⁺ = PA⁺P`, nên

$$\tilde{\Delta} = A^{+}PH_m^\top R = P\,(A^{+}PH_m^\top R)$$

tức **P là left factor** (không phải right factor như §Step 4 viết). Do đó chỉ
`Mᵀh_b = Bᵀ(Ph_b) ≈ 0` có guarantee, còn `Mh_b` thì không. Code inference dùng
`last_hidden @ M = Mᵀh` nên **đúng**; manuscript Eq.(1) `h+αMh` phải sửa thành `h+αMᵀh`.


---
### [B] Q-space thay cho P — khắc phục lỗi *"P không idempotent"*

Lỗi `AssertionError: P không idempotent: rel_err=2.31e-3` **không phải dữ liệu hỏng**
mà là **TF32** (mantissa 10-bit, eps≈5e-4) trên GPU Ampere/Hopper/Blackwell. Thay vì
nới assert, v4 **không dựng P**: làm toàn bộ đại số trong không gian null `k` chiều với
`Q ∈ ℝ^{d×k}` trực chuẩn, `Y = HₘQ`:

$$w = (Y^\top Y + \lambda I_k)^{-1} Y^\top \mathbf{1}, \quad u = Qw, \quad \mathbf{h}' = \mathbf{h} + \alpha(u^\top\mathbf{h})\,\mathbf{r}$$

`cond` giảm từ ~3e20 (suy biến, buộc pinv) xuống ~2.7e3 (SPD, Cholesky), và
`u ∈ range(Q)` trở thành tính chất **đại số** thay vì "hy vọng số học".


---
### [C] Ablation — cô lập đúng vai trò của null-space

`M = u·rᵀ` luôn rank-1 ⇒ cả full lẫn ablation đều là `h' = h + α(uᵀh)r`, khác **đúng
một biến**: `u` có bị ép vào `range(Q)` hay không. `selectivity = |gate(mal)|/|gate(ben)|`
đo mức chọn lọc. Lưu ý **`λ` không dùng chung được**: ở full, `λ‖PΔ̃‖²` chỉ phạt trong
`range(P)` và phổ đã bị cắt; ở P=I phổ `HᵀH` giữ nguyên nên cùng `λ=10` gần như không
regularize (`λ/λ_max ≈ 2e-4`). Phải sweep `λ` riêng cho ablation.


---
### Cell kiểm chứng (chạy được độc lập)

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# KIỂM CHỨNG NUMERIC CHO AGOPNs  (gộp verify_math + verify_qspace + verify_ablation)
# ═══════════════════════════════════════════════════════════════════════════════
# Cell tự chứa. Chạy trong repo AGOPNullSpace: tự import src/utils/steering_utils.py.
# Ba phần:
#   [A] verify_math     — P là LEFT factor; Mᵀh_b≈0 (code) vs Mh_b≉0 (Eq.1 manuscript);
#                         M rank-1; bfloat16 phá triệt tiêu benign.
#   [B] verify_qspace   — đường Q-space ≡ đường ma trận; u∈range(Q) theo cấu trúc;
#                         TF32 làm hỏng assert cũ; leakage + selectivity ở quy mô thật.
#   [C] verify_ablation — selectivity FULL vs IDENTITY(P=I) vs VECTOR(h+αr); λ không
#                         dùng chung được giữa full và ablation.
# ───────────────────────────────────────────────────────────────────────────────
import os, sys, time
import torch

# ── Import steering_utils của repo (fallback: định nghĩa tối thiểu tại chỗ) ─────
_HERE = os.getcwd()
for _p in [os.path.join(_HERE, "src"), os.path.join(_HERE, "..", "src"), _HERE]:
    if os.path.isdir(os.path.join(_p, "utils")):
        sys.path.insert(0, _p); break
try:
    from utils.steering_utils import (
        disable_tf32, null_space_basis_l, cal_steering_factors_q,
        cal_steering_factors_ridge_l, cal_tilde_delta_with_regularization_l,
        benign_leakage, steering_signal_stats,
    )
    print("✓ dùng src/utils/steering_utils.py của repo")
except Exception as _e:
    print(f"⚠ không import được repo steering_utils ({_e}); dùng bản rút gọn nội bộ")
    def disable_tf32(verbose=True):
        for s in (lambda: setattr(torch.backends.cuda.matmul, "allow_tf32", False),
                  lambda: setattr(torch.backends.cudnn, "allow_tf32", False),
                  lambda: torch.set_float32_matmul_precision("highest")):
            try: s()
            except Exception: pass
    def null_space_basis_l(A, abs_nullspace_ratio=0.0, min_null_space_ratio=0.1,
                           dtype=None, verbose=True):
        if dtype is not None: A = A.to(dtype)
        N, d = A.shape
        _, S, Vh = torch.linalg.svd(A, full_matrices=False)
        num = int(d * abs_nullspace_ratio) if abs_nullspace_ratio > 0 else int(d * min_null_space_ratio)
        num = max(1, min(num, d))
        if verbose:
            e = (S[-num:]**2).sum()/(S**2).sum()
            print(f"  null space: k={num}/{d} (ρ={num/d:.3f}) benign energy={e:.3e}")
        return Vh[-num:, :].T.conj().contiguous()
    def cal_steering_factors_q(H, Q, r, lambda_reg, device="cpu"):
        H, Q, r = H.float(), Q.float(), r.float()
        Y = H @ Q; G = Y.T @ Y; G.diagonal().add_(lambda_reg); G = 0.5*(G+G.T)
        w = torch.cholesky_solve(Y.sum(0, keepdim=True).T, torch.linalg.cholesky(G)).squeeze(1)
        return Q @ w, r
    def cal_steering_factors_ridge_l(H, r, lambda_reg, device="cpu"):
        H, r = H.float(), r.float()
        A = H.T @ H; A.diagonal().add_(lambda_reg); A = 0.5*(A+A.T)
        u = torch.cholesky_solve(H.sum(0, keepdim=True).T, torch.linalg.cholesky(A)).squeeze(1)
        return u, r
    def cal_tilde_delta_with_regularization_l(H, P, r, lambda_reg, device="cpu"):
        H, P, r = H.float(), P.float(), r.float()
        X = H @ P; XtX = X.T@X
        A = 0.5*((XtX + lambda_reg*P) + (XtX + lambda_reg*P).T)
        return torch.linalg.pinv(A, hermitian=True) @ torch.outer(X.sum(0), r)
    def benign_leakage(H, Q, max_n=4000):
        H = H[:max_n].float(); ratio = (H@Q).norm(dim=1)/H.norm(dim=1).clamp(min=1e-12)
        return {"leakage_mean": float(ratio.mean()), "leakage_p95": float(ratio.quantile(0.95)),
                "leakage_max": float(ratio.max())}
    def steering_signal_stats(H, u, r, max_n=2000):
        H = H[:max_n].float(); g = H@u; s = g.abs()*r.norm()
        return {"gate_mean": float(g.mean()), "gate_std": float(g.std()),
                "mean_signal_norm": float(s.mean())}

torch.manual_seed(0)
disable_tf32(verbose=False)
DEV = "cpu"   # đổi "cuda" nếu muốn kiểm chứng luôn trên GPU (TF32 đã tắt)


# ═══════════════════════════════════════════════════════════════════════════════
# [A] verify_math — left/right factor, rank-1, bfloat16
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*80 + "\n[A] LEFT vs RIGHT FACTOR — Mᵀh_b≈0 (code) vs Mh_b≉0 (Eq.1)\n" + "="*80)
d, N_b, N_m, LAM = 128, 900, 300, 10.0
basis = torch.linalg.qr(torch.randn(d, 50))[0]
Hb = (torch.randn(N_b,50)@basis.T + .05*torch.randn(N_b,d)).double()
Hm = (torch.randn(N_m,50)@basis.T + .05*torch.randn(N_m,d) + .8*torch.randn(1,d)).double()
r = torch.randn(d).double(); r /= r.norm()

_, S, Vh = torch.linalg.svd(Hb, full_matrices=False)
Q = Vh[-int(d*0.6):, :].T.conj(); P = (Q @ Q.T).double()
X = Hm @ P
A = 0.5*((X.T@X + LAM*P) + (X.T@X + LAM*P).T)
D = torch.linalg.pinv(A, hermitian=True) @ torch.outer(X.sum(0), r)   # Δ̃
M = P @ D

print(f"  ||PΔ̃ − Δ̃||/||Δ̃|| = {((P@D-D).norm()/D.norm()):.2e}   → P là LEFT factor")
print(f"  ||Δ̃P − Δ̃||/||Δ̃|| = {((D@P-D).norm()/D.norm()):.2e}   → P KHÔNG là right factor")
print(f"  (manuscript §Step 4 viết Δ̃=A·P tức right factor ⇒ SAI)")
mh, mth = (P@D@Hb.T).T, Hb@M
print(f"\n  Eq.(1) h+αMh  : mean||M h_b||  = {mh.norm(dim=1).mean():.3e}   ← KHÔNG ≈ 0")
print(f"  code  h+α·hM  : mean||Mᵀh_b|| = {mth.norm(dim=1).mean():.3e}   ← ≈ 0 ✓")
mtm = Hm@M
print(f"  malicious     : mean||Mᵀh_m|| = {mtm.norm(dim=1).mean():.3f}  cos(·,r)="
      f"{torch.nn.functional.cosine_similarity(mtm, r.expand_as(mtm), dim=1).mean():.3f}")

sv = torch.linalg.svdvals(M)
print(f"\n  M rank-1?  σ₂/σ₁ = {sv[1]/sv[0]:.2e}   ||M − u⊗r||/||M|| = "
      f"{(M - torch.outer(M@r, r)).norm()/M.norm():.2e}   (Mᵀh = (uᵀh)·r)")

print(f"\n  bfloat16 (v2 cast steering_matrix → bf16):")
M32 = M.float()
for dt, nm in [(torch.float32,"fp32"), (torch.bfloat16,"bf16")]:
    out = (Hb.float().to(dt) @ M32.to(dt)).float(); ref = Hb.float() @ M32
    print(f"    {nm}: mean||Mᵀh_b|| = {out.norm(dim=1).mean():.3e}  "
          f"rel_err vs fp32 = {((out-ref).norm()/ref.norm()):.2e}")


# ═══════════════════════════════════════════════════════════════════════════════
# [B] verify_qspace — Q-space ≡ matrix, u∈range(Q), TF32, leakage/selectivity
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*80 + f"\n[B] Q-SPACE ở quy mô thật (d=4096, ρ=0.6, k=2457)\n" + "="*80)
d, N_b, N_bho, N_m, RHO, LAM = 4096, 3200, 800, 2000, 0.6, 10.0
basis = torch.linalg.qr(torch.randn(d, 800))[0]
mk = lambda n: torch.randn(n,800)@basis.T + 0.05*torch.randn(n,d)
Hb, Hbho, Hm = mk(N_b), mk(N_bho), mk(N_m)+0.6*torch.randn(1,d)
r = torch.randn(d); r /= r.norm()

t = time.time()
Q = null_space_basis_l(Hb, abs_nullspace_ratio=RHO)
u, _ = cal_steering_factors_q(Hm, Q, r, lambda_reg=LAM, device=DEV)
print(f"  [Q-space] {time.time()-t:.1f}s  ||u||={u.norm():.4f}")

# tương đương đường ma trận cũ, trên fp64
Qd = null_space_basis_l(Hb.double(), abs_nullspace_ratio=RHO, verbose=False)
ud, _ = cal_steering_factors_q(Hm.double(), Qd, r.double(), lambda_reg=LAM, device=DEV)
Pd = Qd@Qd.T; Xd = Hm.double()@Pd
Ad = 0.5*((Xd.T@Xd + LAM*Pd) + (Xd.T@Xd + LAM*Pd).T)
u_mat = Pd @ (torch.linalg.pinv(Ad, hermitian=True) @ Xd.sum(0))
print(f"  [1] fp64: ||u_qspace − u_matrix||/||u|| = {(ud-u_mat).norm()/ud.norm():.2e}   ← cùng nghiệm")
print(f"  [2] u ngoài range(Q): ||u−QQᵀu||/||u|| = {(u-Q@(Q.T@u)).norm()/u.norm():.2e}   ← ~0 theo CẤU TRÚC")
print(f"      residual gate: mean((H_m·u−1)²) = {((Hm@u-1)**2).mean():.2e}")

tf = lambda x: (x.contiguous().view(torch.int32) & torch.tensor(-0x2000, dtype=torch.int32)).view(torch.float32)
Qt = tf(Q); Pt = Qt@Qt.T; idx = torch.randperm(d)[:256]; sub = Pt[idx][:,idx]
idem = ((Pt[:,idx].T@Pt[:,idx]) - sub).norm()/sub.norm()
print(f"  [3] TF32 mô phỏng: v3 assert P²=P (sub-block) = {idem:.2e}  "
      f"→ {'FAIL >1e-3 (đúng lỗi bạn gặp)' if idem>1e-3 else 'may mắn pass lần này'}")
print(f"      v4 tính u qua Q ⇒ miễn nhiễm (u∈range(Q) theo đại số)")

leak = benign_leakage(Hbho, Q)
sm, sb = steering_signal_stats(Hm, u, r), steering_signal_stats(Hbho, u, r)
print(f"  [4] leakage benign held-out ||Qᵀh||/||h||: mean={leak['leakage_mean']:.4f} p95={leak['leakage_p95']:.4f}")
print(f"      gate uᵀh: malicious={sm['gate_mean']:+.4f}±{sm['gate_std']:.4f}  "
      f"benign={sb['gate_mean']:+.5f}±{sb['gate_std']:.5f}")
print(f"      selectivity ||Mᵀh_m||/||Mᵀh_b|| = {sm['mean_signal_norm']/max(sb['mean_signal_norm'],1e-9):.0f}x")


# ═══════════════════════════════════════════════════════════════════════════════
# [C] verify_ablation — FULL vs IDENTITY(P=I) vs VECTOR; λ không dùng chung
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*80 + "\n[C] ABLATION selectivity = |gate(malicious)| / |gate(benign)|\n" + "="*80)
d, N_b, N_bho, N_m, RHO, LAM_NS = 256, 900, 300, 300, 0.6, 10.0
basis = torch.linalg.qr(torch.randn(d, 50))[0]
mk = lambda n: torch.randn(n,50)@basis.T + 0.05*torch.randn(n,d)
Hb, Hbho, Hm = mk(N_b), mk(N_bho), mk(N_m)+0.8*torch.randn(1,d)
r = torch.randn(d); r /= r.norm()

def rep(u, name, lam):
    gm, gb = Hm@u, Hbho@u
    print(f"    {name:<9} λ={lam:<7.3g} gate(mal)={gm.mean():+.4f}±{gm.std():.4f}  "
          f"gate(ben)={gb.mean():+.5f}±{gb.std():.5f}  selectivity={gm.abs().mean()/gb.abs().mean():8.1f}x")

Q = null_space_basis_l(Hb, abs_nullspace_ratio=RHO, verbose=False)
u_full, _ = cal_steering_factors_q(Hm, Q, r, lambda_reg=LAM_NS, device=DEV)
print(f"  FULL (null-space):  u∈range(Q) err = {(u_full-Q@(Q.T@u_full)).norm()/u_full.norm():.1e}")
rep(u_full, "FULL", LAM_NS)
print()
lam_max = torch.linalg.eigvalsh(Hm.T@Hm)[-1].item()
print(f"  IDENTITY (P=I):  λ_max(HᵀH)={lam_max:.3g} ⇒ λ=10 cho λ/λ_max={10/lam_max:.1e} "
      f"(≈0: KHÔNG regularize) → phải tune λ RIÊNG, không dùng chung với FULL")
for lam in [10.0, 1e2, 1e3, 1e4]:
    u, _ = cal_steering_factors_ridge_l(Hm, r, lambda_reg=lam, device=DEV)
    rep(u, "IDENTITY", lam)
print(f"\n    {'VECTOR':<9} λ={'-':<7} h+α·r cho MỌI prompt "
      f"                                 selectivity=     1.0x (định nghĩa)")
print("\n" + "="*80)
print("KẾT LUẬN: code inference (Mᵀh) ĐÚNG; manuscript Eq.(1)/§Step4 cần sửa sang Mᵀh.")
print("          M rank-1 ⇒ additive steering với cổng uᵀh (O(d), 70B: 7GB→1.7MB).")
print("          Lỗi 'P không idempotent' = TF32; v4 (Q-space) khắc phục tận gốc.")
print("="*80)


✓ dùng src/utils/steering_utils.py của repo

[A] LEFT vs RIGHT FACTOR — Mᵀh_b≈0 (code) vs Mh_b≉0 (Eq.1)
  ||PΔ̃ − Δ̃||/||Δ̃|| = 2.11e-13   → P là LEFT factor
  ||Δ̃P − Δ̃||/||Δ̃|| = 6.93e-01   → P KHÔNG là right factor
  (manuscript §Step 4 viết Δ̃=A·P tức right factor ⇒ SAI)

  Eq.(1) h+αMh  : mean||M h_b||  = 8.006e-02   ← KHÔNG ≈ 0
  code  h+α·hM  : mean||Mᵀh_b|| = 5.533e-03   ← ≈ 0 ✓
  malicious     : mean||Mᵀh_m|| = 0.999  cos(·,r)=1.000

  M rank-1?  σ₂/σ₁ = 5.33e-14   ||M − u⊗r||/||M|| = 1.70e-13   (Mᵀh = (uᵀh)·r)

  bfloat16 (v2 cast steering_matrix → bf16):
    fp32: mean||Mᵀh_b|| = 5.533e-03  rel_err vs fp32 = 0.00e+00
    bf16: mean||Mᵀh_b|| = 5.544e-03  rel_err vs fp32 = 3.07e-02

[B] Q-SPACE ở quy mô thật (d=4096, ρ=0.6, k=2457)

Intel oneMKL ERROR: Parameter 8 was incorrect on entry to SSYEVD.


RuntimeError: false INTERNAL ASSERT FAILED at "/pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp":1601, please report a bug to PyTorch. linalg.eigh: Argument 8 has illegal value. Most certainly there is a bug in the implementation calling the backend library.

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]      = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]   = "7"#"0,1,2,3,4,5,6"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch

def compare(path_a, path_b):
    a = torch.load(path_a, map_location="cuda")
    b = torch.load(path_b, map_location="cuda")
    
    print(f"{'L':>3} {'||a_l||':>10} {'r=|b|/|a|':>10} {'cos':>8} "
          f"{'alpha*':>8} {'resid':>8} {'row_cos':>8}")
    for l in range(a.shape[0]):
        x = a[l].flatten().double()
        y = b[l].flatten().double()
        nx, ny = x.norm(), y.norm()
        if nx == 0 or ny == 0:
            print(f"{l:>3}  ZERO LAYER  (||a||={nx:.2e}, ||b||={ny:.2e})")
            continue
        dot   = x @ y
        cos   = (dot / (nx * ny)).item()
        alpha = (dot / nx**2).item()              # scale tối ưu
        resid = ((y - alpha * x).norm() / ny).item()  # còn lại sau khi bỏ scale
        rc    = torch.nn.functional.cosine_similarity(
                    a[l].double(), b[l].double(), dim=-1).mean().item()
        print(f"{l:>3} {nx.item():>10.3e} {(ny/nx).item():>10.4f} "
              f"{cos:>8.4f} {alpha:>8.4f} {resid:>8.4f} {rc:>8.4f}")

In [8]:
a = "./data/steering_matrix/steering_matrix_llama3.1_agopn_rfm.pt"
b = "./data/steering_matrix/steering_matrix_llama3.1_agopn_rfm.pt"
compare(a, b)

  L    ||a_l||  r=|b|/|a|      cos   alpha*    resid  row_cos
  0  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  1  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  2  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  3  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  4  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  5  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  6  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  7  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  8  5.458e+00     1.0000   1.0000   1.0000   0.0000   1.0000
  9  5.331e+00     1.0000   1.0000   1.0000   0.0000   1.0000
 10  4.905e+00     1.0000   1.0000   1.0000   0.0000   1.0000
 11  4.764e+00     1.0000   1.0000   1.0000   0.0000   1.0000
 12  5.250e+00     1.0000   1.0000   1.0000   0.0000   1.0000
 13  4.807e+00     1.0000   1.0000   1.0000   0.0000   1.0000
 14  4.279e+00     1.0000   1.0000   1.0000   0.0000   1.0000
 15  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
 16  3.855e+00     1.0000   1.0000   1.0000   0.0000  

In [6]:
a = "./data/steering_matrix/steering_matrix_llama3.1_agopn_rfm.pt"
b = "./data/steering_matrix/steering_matrix_llama3.1_dim.pt"
compare(a, b)

  L    ||a_l||  r=|b|/|a|      cos   alpha*    resid  row_cos
  0  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  1  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  2  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  3  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  4  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  5  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  6  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  7  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  8  5.458e+00     1.5766  -0.0410  -0.0647   0.9992  -0.0353
  9  5.331e+00     1.7103  -0.0442  -0.0756   0.9990  -0.0388
 10  4.905e+00     2.0363  -0.1798  -0.3662   0.9837  -0.1558
 11  4.764e+00     2.2258  -0.1647  -0.3665   0.9863  -0.1422
 12  5.250e+00     3.1161  -0.0111  -0.0345   0.9999  -0.0094
 13  4.807e+00     3.0605  -0.3591  -1.0991   0.9333  -0.3080
 14  4.279e+00     3.3726  -0.1367  -0.4609   0.9906  -0.1178
 15  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
 16  3.855e+00     4.2531  -0.0868  -0.3691   0.9962  

In [7]:
a = "./data/steering_matrix/steering_matrix_llama3.1_agopn_rfm.pt"
b = "./data/steering_matrix/steering_matrix_llama3.1_agopn_linear.pt"
compare(a, b)

  L    ||a_l||  r=|b|/|a|      cos   alpha*    resid  row_cos
  0  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  1  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  2  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  3  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  4  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  5  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  6  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  7  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  8  5.458e+00     1.0000   0.6364   0.6364   0.7713   0.6364
  9  5.331e+00     1.0000   0.7460   0.7460   0.6659   0.7460
 10  4.905e+00     1.0000   0.3309   0.3309   0.9437   0.3309
 11  4.764e+00     1.0000   0.6390   0.6390   0.7692   0.6390
 12  5.250e+00     1.0000  -0.0099  -0.0099   1.0000  -0.0099
 13  4.807e+00     1.0000   0.3469   0.3469   0.9379   0.3469
 14  4.279e+00     1.0000  -0.2448  -0.2448   0.9696  -0.2448
 15  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
 16  3.855e+00     1.0000  -0.3660  -0.3660   0.9306  

In [ ]:
python src/calc_steering_matrix_rfm.py \
    --model_name llama3.1 --embedding_dir data/embeddings/llama3.1 \
    --save_path data/steering_matrix/steering_matrix_llama3.1_agopn_rfm_data1hh.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:0"  

python src/calc_steering_matrix_rfm.py \
    --model_name llama3.1 --embedding_dir data/embeddings/llama3.1 \
    --save_path data/steering_matrix/steering_matrix_llama3.1_agopn_rfm.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device cuda  

python src/calc_steering_matrix_rfm_no_nullspace.py \
    --model_name llama3.1 --embedding_dir data/embeddings/llama3.1 \
    --save_path data/steering_matrix/steering_matrix_llama3.1_agopn_rfm_no_nullspace.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device cuda  

python src/calc_steering_matrix_rfm.py \
    --model_name llama3.1 --embedding_dir data/embeddings/llama3.1 \
    --save_path data/steering_matrix/steering_matrix_llama3.1_agopn_linear.pt \
    --probe linear --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device cuda   

python src/calc_steering_matrix.py --model_name llama3.1 \
                                    --embedding_dir data/embeddings/llama3.1 \
                                    --device cuda \
                                    --save_path data/steering_matrix/steering_matrix_llama3.1_dim.pt

In [ ]:
python src/calc_steering_matrix_rfm_no_nullspace.py \
    --model_name qwen2.5 --embedding_dir data/embeddings/qwen2.5 \
    --save_path data/steering_matrix/steering_matrix_qwen2.5_agopn_rfm_data1hh_no_nullspace.pt \
    --probe rfm --rfm_iters 10 --n_components 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:1"    

python src/calc_steering_matrix_rfm_hh.py \
    --model_name qwen2.5 --embedding_dir data/embeddings/qwen2.5 \
    --save_path data/steering_matrix/steering_matrix_qwen2.5_agopn_rfm_data1hh.pt \
    --probe rfm --rfm_iters 10 --n_components 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:1"     

python src/calc_steering_matrix_rfm.py \
    --model_name qwen2.5 --embedding_dir data/embeddings/qwen2.5 \
    --save_path data/steering_matrix/steering_matrix_qwen2.5_agopn_rfm.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device cuda 

python src/calc_steering_matrix_rfm_no_nullspace.py \
    --model_name qwen2.5 --embedding_dir data/embeddings/qwen2.5 \
    --save_path data/steering_matrix/steering_matrix_qwen2.5_agopn_rfm_no_nullspace.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:1"   

python src/calc_steering_matrix_rfm.py \
    --model_name qwen2.5 --embedding_dir data/embeddings/qwen2.5 \
    --save_path data/steering_matrix/steering_matrix_qwen2.5_agopn_linear.pt \
    --probe linear --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device cuda   

python src/calc_steering_matrix.py --model_name qwen2.5 \
                                    --embedding_dir data/embeddings/qwen2.5 \
                                    --device cuda \
                                    --save_path data/steering_matrix/steering_matrix_qwen2.5_dim.pt

In [ ]:
python src/calc_steering_matrix_rfm_hh.py \
    --model_name gemma2 --embedding_dir data/embeddings/gemma2 \
    --save_path data/steering_matrix/steering_matrix_gemma2_agopn_rfm_data1hh.pt \
    --probe rfm --rfm_iters 10 --n_components 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:2" 

python src/calc_steering_matrix_rfm.py \
    --model_name gemma2 --embedding_dir data/embeddings/gemma2 \
    --save_path data/steering_matrix/steering_matrix_gemma2_agopn_rfm.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:1"   

python src/calc_steering_matrix_rfm_no_nullspace.py \
    --model_name gemma2 --embedding_dir data/embeddings/gemma2 \
    --save_path data/steering_matrix/steering_matrix_gemma2_agopn_rfm_no_nullspace.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:2"   

python src/calc_steering_matrix_rfm.py \
    --model_name gemma2 --embedding_dir data/embeddings/gemma2 \
    --save_path data/steering_matrix/steering_matrix_gemma2_agopn_linear.pt \
    --probe linear --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:2"   

python src/calc_steering_matrix.py --model_name gemma2 \
                                    --embedding_dir data/embeddings/gemma2 \
                                    --device cuda \
                                    --save_path data/steering_matrix/steering_matrix_gemma2_dim.pt

In [ ]:
python src/calc_steering_matrix_rfm.py \
    --model_name llama3.1 --embedding_dir data/embeddings/llama3.1 \
    --save_path data/steering_matrix/steering_matrix_llama3.1_agopn_rfm_data1hh.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:0"  

python src/calc_steering_matrix_rfm.py \
    --model_name qwen2.5 --embedding_dir data/embeddings/qwen2.5 \
    --save_path data/steering_matrix/steering_matrix_qwen2.5_agopn_rfm_data1hh.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:1" 

python src/calc_steering_matrix_rfm.py \
    --model_name gemma2 --embedding_dir data/embeddings/gemma2 \
    --save_path data/steering_matrix/steering_matrix_gemma2_agopn_rfm_data1hh.pt \
    --probe rfm --rfm_iters 5 --tuning_metric auc --lambda_reg 10.0 \
    --holdout_benign 1000 --device "cuda:2" 

python src/calc_steering_matrix_dim_hh.py \
    --model_name llama3.1 \
    --embedding_dir data/embeddings/llama3.1 \
    --save_path data/steering_matrix/steering_matrix_llama3.1_dim_data1hh.pt \
    --device "cuda:3"  

python src/calc_steering_matrix_dim_hh.py \
    --model_name qwen2.5 \
    --embedding_dir data/embeddings/qwen2.5 \
    --save_path data/steering_matrix/steering_matrix_qwen2.5_dim_data1hh.pt \
    --device "cuda:4"  

python src/calc_steering_matrix_dim_hh.py \
    --model_name gemma2 \
    --embedding_dir data/embeddings/gemma2 \
    --save_path data/steering_matrix/steering_matrix_gemma2_dim_data1hh.pt \
    --device "cuda:5"  

In [ ]:
python ./src/prepare_harmful_harmless_instruction_embeddings.py --model_name_or_path meta-llama/Llama-3.1-8B-Instruct --output_dir ./data/embeddings/llama3.1/harmful_harmless_instructions --split train --batch_size 2 --device "cuda:0" --dtype bfloat16 --save_separated
 
python ./src/prepare_harmful_harmless_instruction_embeddings.py --model_name_or_path Qwen/Qwen2.5-7B-Instruct --output_dir ./data/embeddings/qwen2.5/harmful_harmless_instructions --split train --batch_size 2 --device "cuda:1" --dtype bfloat16 --save_separated
 
python ./src/prepare_harmful_harmless_instruction_embeddings.py --model_name_or_path google/gemma-2-9b-it --output_dir ./data/embeddings/gemma2/harmful_harmless_instructions --split train --batch_size 2 --device "cuda:2" --dtype bfloat16 --save_separated

In [ ]:
f"{STEERING_MATRIX_DIR}/steering_matrix_llama3.1_agopn_linear.pt",
f"{STEERING_MATRIX_DIR}/steering_matrix_llama3.1_agopn_rfm.pt",
f"{STEERING_MATRIX_DIR}/steering_matrix_llama3.1_dim.pt",

f"{STEERING_MATRIX_DIR}/steering_matrix_qwen2.5_agopn_linear.pt",
f"{STEERING_MATRIX_DIR}/steering_matrix_qwen2.5_agopn_rfm.pt",
f"{STEERING_MATRIX_DIR}/steering_matrix_qwen2.5_dim.pt",

f"{STEERING_MATRIX_DIR}/steering_matrix_gemma2_agopn_linear.pt",
f"{STEERING_MATRIX_DIR}/steering_matrix_gemma2_agopn_rfm.pt",
f"{STEERING_MATRIX_DIR}/steering_matrix_gemma2_dimpt",


In [5]:
a = "./data/steering_matrix/steering_matrix_llama3.1_dim_data1hh.pt"
b = "./data/steering_matrix/steering_matrix_llama3.1_agopn_rfm_data1hh.pt"
compare(a, b)

  L    ||a_l||  r=|b|/|a|      cos   alpha*    resid  row_cos
  0  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  1  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  2  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  3  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  4  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  5  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  6  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  7  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
  8  5.458e+00     1.0066   0.2061   0.2074   0.9785   0.1581
  9  5.331e+00     1.0100   0.2257   0.2279   0.9742   0.1735
 10  4.905e+00     1.0267   0.2358   0.2421   0.9718   0.1716
 11  4.764e+00     1.0369   0.2913   0.3020   0.9566   0.2076
 12  5.250e+00     1.0149   0.2454   0.2490   0.9694   0.1704
 13  4.807e+00     0.9974   0.2060   0.2054   0.9786   0.1471
 14  4.279e+00     1.0026   0.2095   0.2101   0.9778   0.1483
 15  ZERO LAYER  (||a||=0.00e+00, ||b||=0.00e+00)
 16  3.855e+00     1.0376   0.1893   0.1964   0.9819  